In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from utils.spark_session import get_spark

In [ ]:
HDFS_ROOT   = "hdfs://localhost:9000/rossmann"
OUTPUT_PATH = f"{HDFS_ROOT}/clean/df_clean.parquet"

In [ ]:
def load_data(spark):
    """Đọc sale.csv và store.csv từ HDFS, trả về 2 pandas DataFrame."""
    df_sales_spark  = spark.read.csv(f"{HDFS_ROOT}/sale.csv",
                                     header=True, inferSchema=True)
    df_stores_spark = spark.read.csv(f"{HDFS_ROOT}/store.csv",
                                     header=True, inferSchema=True)

    print(f"Sales  - {df_sales_spark.count():,} rows x {len(df_sales_spark.columns)} cols")
    print(f"Stores - {df_stores_spark.count():,} rows x {len(df_stores_spark.columns)} cols")

    df_sales  = df_sales_spark.toPandas()
    df_stores = df_stores_spark.toPandas()
    return df_sales, df_stores


In [ ]:
def clean_and_join(df_sales: pd.DataFrame,
                   df_stores: pd.DataFrame) -> pd.DataFrame:
    """Làm sạch missing values và join 2 bộ dữ liệu."""
    print("\n── Bộ dữ liệu Sales ──")
    print(df_sales.shape)
    print(df_sales.isnull().sum())

    print("\n── Bộ dữ liệu Stores ──")
    print(df_stores.shape)
    print(df_stores.isnull().sum())

    # Nhận thấy rằng chỉ có 3 mẫu trong biến CompetitionDistance là bị missing,
    # ta xử lý nó bằng cách fill giá trị trung bình của biến CompetitionDistance
    df_stores["CompetitionDistance"] = df_stores[
        "CompetitionDistance"
    ].fillna(df_stores["CompetitionDistance"].mean())

    # Những mẫu nào không có giá trị trong biến Promo2 thì ta sẽ fill bằng 0
    df_stores["Promo2"] = df_stores["Promo2"].fillna(0)

    # Join theo Store
    df_sales["Date"] = pd.to_datetime(df_sales["Date"])
    df = df_sales.merge(right=df_stores, on="Store", how="left")
    df["Date"] = pd.to_datetime(df["Date"])

    print(f"\n✅ Joined shape: {df.shape}")
    print(df.head())
    return df

In [ ]:
def run_eda(df: pd.DataFrame) -> None:
    """EDA – 4 biểu đồ trực quan hoá."""
    df_open = df[df["Sales"] > 0].copy()

    # ── Biểu đồ 1: Phân phối Sales ───────────────────────────────────────────
    plt.figure(figsize=(12, 8))
    plt.hist(df_open["Sales"], bins=50)
    plt.title("Phân phối của Sales")
    plt.xlabel("Sales")
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()

    # ── Biểu đồ 2: Sales trung bình theo ngày + ngày có Promo ────────────────
    daily      = df_open.groupby("Date")["Sales"].mean()
    promo_days = df_open[df_open["Promo"] == 1].groupby("Date")["Sales"].mean()

    plt.figure(figsize=(14, 4))
    plt.plot(daily.index, daily.values, color="steelblue", linewidth=1)
    plt.scatter(promo_days.index, promo_days.values,
                color="red", s=5, label="Có Promo")
    plt.title("Giá trị trung bình của Sales theo ngày", weight="bold")
    plt.legend()
    plt.tight_layout()
    plt.show()

    # ── Biểu đồ 3: Sales theo Ngày trong tuần & Tháng ────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    dow = df_open.groupby("DayOfWeek")["Sales"].mean()
    axes[0].bar(["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"],
                dow.values, color="steelblue")
    axes[0].set_title("Sales theo Ngày trong Tuần", weight="bold")
    axes[0].set_ylabel("Giá trị Sale trung bình", weight="bold")

    month = df_open.groupby(df_open["Date"].dt.month)["Sales"].mean()
    axes[1].bar(range(1, 13), month.values, color="green")
    axes[1].set_title("Sales theo Tháng", weight="bold")
    axes[1].set_xlabel("Tháng", weight="bold")

    plt.tight_layout()
    plt.show()

    # ── Biểu đồ 4: Heatmap tương quan ────────────────────────────────────────
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    corr = df[numeric_cols].corr()

    plt.figure(figsize=(14, 8))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm")
    plt.title("Ma trận tương quan giữa các biến", weight="bold")
    plt.tight_layout()
    plt.show()

    print("\nTop tương quan với Sales:")
    print(corr["Sales"].drop("Sales").sort_values(ascending=False))

In [ ]:
def export_clean(df: pd.DataFrame, spark) -> None:
    """Lưu df sạch ra HDFS dạng parquet để file 02 đọc tiếp."""
    df_export = df.copy()
    df_export["Date"] = df_export["Date"].astype(str)
    for col in df_export.select_dtypes(include=[object]).columns:
        df_export[col] = df_export[col].astype(str)
    sdf = spark.createDataFrame(df_export)
    sdf.write.mode("overwrite").parquet(OUTPUT_PATH)
    print(f"\n Đã lưu df_clean → {OUTPUT_PATH}")


In [ ]:
if __name__ == "__main__":
    spark = get_spark("01_Data_EDA")

    df_sales, df_stores = load_data(spark)
    df = clean_and_join(df_sales, df_stores)
    run_eda(df)
    export_clean(df, spark)

    spark.stop()
    print("\n[01] Done.")
